# Proyecto 2 — Diseño generativo de personajes
## Universo *Lumen: Guardianes Elementales* · DCGAN 64×64

Deep Learning 2026 — Universidad del Valle.

Este notebook orquesta el pipeline completo usando los módulos de `src/`. Puede
ejecutarse de extremo a extremo (entrena ~45 min en GPU/MPS) o cargar los
checkpoints ya entrenados para regenerar las evidencias y la galería.

**Semillas:** `SEED_DATA=20260101`, `SEED_TRAIN=42`, `SEED_EVAL=7`, `GALLERY_SEED=2026`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import config as C
print('Dispositivo:', C.get_device())

## 1. Universo y datos
Ver `docs/universo.md`. El dataset se genera de forma procedural (creación propia,
CC0) y es totalmente reproducible desde `SEED_DATA`.

In [ ]:
from src.data import generar_dataset
cfg = C.base_config()
generar_dataset(C.IMG_DIR, cfg.dataset_size, cfg.image_size, C.SEED_DATA, C.SAMPLE_DIR)
from PIL import Image
Image.open(os.path.join(C.SAMPLE_DIR, 'dataset_muestra_8x8.png'))

## 2. Entrenamiento (base + experimentos)
Tres corridas que cambian **una sola cosa a la vez** (ver `docs/hipotesis.md`).
Si ya existen los checkpoints, puede saltar esta celda.

In [ ]:
from src.experiments import entrenar_todo, comparar
entrenar_todo()   # base, hinge, spectral
comparar()        # figuras de comparación de los experimentos

## 3. Evidencia 1 — Rejilla de ruido fijo (evolución)

In [ ]:
Image.open(os.path.join(C.EVID_DIR, 'base', 'evolucion_ruido_fijo.png'))

## 4. Evidencia 2 — Curvas de pérdida interpretadas
En una GAN sana las curvas **no bajan juntas**: se estabilizan enfrentadas. Una
pérdida de D cercana a 0 por muchas épocas sería señal de alarma.

In [ ]:
Image.open(os.path.join(C.EVID_DIR, 'base', 'curvas_perdida.png'))

## 5. Galería final (10 personajes) + tasa de selección

In [ ]:
from src.gallery import generar_galeria
info = generar_galeria('base')
print('Tasa de selección:', info['tasa_seleccion'], '| criterio:', info['criterio'])
Image.open(os.path.join(C.GALLERY_DIR, 'galeria_10.png'))

## 6. Evidencia 3 — Vecinos más cercanos
Cada personaje junto a su imagen de entrenamiento más parecida (features de CNN).

In [ ]:
from src.nearest_neighbor import prueba_vecinos
prueba_vecinos('base')
Image.open(os.path.join(C.EVID_DIR, 'vecinos_mas_cercanos.png'))